In [2]:

!pip install reportlab

# IMPORTS

from transformers import pipeline, set_seed, GPT2LMHeadModel, GPT2Tokenizer
import torch
import pandas as pd

from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors


# SETUP

set_seed(42)

device = 0 if torch.cuda.is_available() else -1
torch_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

generator = pipeline(
    "text-generation",
    model="gpt2",
    device=device
)

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2").to(torch_device)
tokenizer.pad_token = tokenizer.eos_token

prompts = [
    "A robot found a diary that could predict the future,",
    "Humans colonized Mars and discovered a hidden civilization,",
    "An AI started dreaming and created a new universe,",
    "The internet suddenly became alive and started talking to humans,",
    "A spaceship reached a planet where time flows backwards,"
]


# CREATIVITY FUNCTION

def creativity_score(text):
    words = text.split()
    if len(words) == 0:
        return 0
    return len(set(words)) / len(words)

# STORAGE

results_log = []

for prompt in prompts:

    print("\n" + "="*80)
    print("PROMPT:\n", prompt)


    # MULTI-SAMPLE GENERATION

    outputs = generator(
        prompt,
        max_new_tokens=60,
        num_return_sequences=3,
        temperature=0.8,
        top_p=0.95,
        do_sample=True,
        pad_token_id=50256
    )

    samples = []
    scores = []

    for i, out in enumerate(outputs):
        text = out["generated_text"]
        score = creativity_score(text)

        samples.append(text)
        scores.append(score)

        print(f"\nSAMPLE {i+1}:\n{text}")
        print("Creativity Score:", round(score, 3))

    avg_score = sum(scores) / len(scores)

    print("\nCREATIVITY SUMMARY")
    print("Scores:", [round(s,3) for s in scores])
    print("Average:", round(avg_score, 3))


    # PIPELINE VS MANUAL

    pipe_out = generator(
        prompt,
        max_new_tokens=60,
        num_return_sequences=1,
        do_sample=True
    )[0]["generated_text"]

    manual_inputs = tokenizer(prompt, return_tensors="pt").to(torch_device)

    manual_out = model.generate(
        **manual_inputs,
        max_new_tokens=60,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        repetition_penalty=1.2
    )

    manual_text = tokenizer.decode(manual_out[0], skip_special_tokens=True)

    print("\nPIPELINE OUTPUT:\n", pipe_out)
    print("\nMANUAL OUTPUT:\n", manual_text)

    print("="*80)


    # SAVE CLEAN STRUCTURED ROW

    results_log.append({
        "prompt": prompt,
        "sample_1": samples[0],
        "sample_2": samples[1],
        "sample_3": samples[2],
        "avg_creativity": avg_score,
        "pipeline_output": pipe_out,
        "manual_output": manual_text
    })


# CREATE CSV

df = pd.DataFrame(results_log)
df.to_csv("gpt2_clean_experiment.csv", index=False)

print("\nCSV Saved: gpt2_clean_experiment.csv")

# CREATE PDF REPORT

pdf_file = "GPT2_Experiment_Report.pdf"
doc = SimpleDocTemplate(pdf_file)

styles = getSampleStyleSheet()
content = []

# Title
content.append(Paragraph("GPT-2 Text Generation Experiment Report", styles["Title"]))
content.append(Spacer(1, 12))

# Summary
summary = f"""
This report analyzes GPT-2 text generation using multiple prompts, sampling strategies, and decoding methods.

Total Prompts: {len(df)}
"""
content.append(Paragraph(summary, styles["Normal"]))
content.append(Spacer(1, 12))

# Table
table_data = [["Prompt", "Avg Creativity"]]

for _, row in df.iterrows():
    table_data.append([
        row["prompt"][:60] + "...",
        round(row["avg_creativity"], 3)
    ])

table = Table(table_data)

table.setStyle(TableStyle([
    ("BACKGROUND", (0,0), (-1,0), colors.grey),
    ("TEXTCOLOR", (0,0), (-1,0), colors.whitesmoke),
    ("GRID", (0,0), (-1,-1), 0.5, colors.black),
    ("FONTNAME", (0,0), (-1,0), "Helvetica-Bold"),
]))

content.append(table)
content.append(Spacer(1, 20))

# Detailed Results
for i, row in df.iterrows():

    content.append(Paragraph(f"Prompt {i+1}: {row['prompt']}", styles["Heading2"]))
    content.append(Spacer(1, 6))

    detail = f"""
    <b>Sample 1:</b> {row['sample_1']} <br/><br/>
    <b>Sample 2:</b> {row['sample_2']} <br/><br/>
    <b>Sample 3:</b> {row['sample_3']} <br/><br/>
    <b>Pipeline Output:</b> {row['pipeline_output']} <br/><br/>
    <b>Manual Output:</b> {row['manual_output']} <br/><br/>
    <b>Average Creativity:</b> {row['avg_creativity']}
    """

    content.append(Paragraph(detail, styles["Normal"]))
    content.append(Spacer(1, 12))


doc.build(content)

print("\nPDF Generated: GPT2_Experiment_Report.pdf")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



PROMPT:
 A robot found a diary that could predict the future,


...


PIPELINE OUTPUT:
 The internet suddenly became alive and started talking to humans, and was able to talk to them in the same way that we have to talk to humans today. We can speak to people, too, and people from different cultures can talk to us in the same way. It's just the way things are, and we still are. In fact, our communication

MANUAL OUTPUT:
 The internet suddenly became alive and started talking to humans, a new age of social media. People were connecting with each other through their own stories (and sometimes the people they connected with).
In 2016, I had an idea for how we could build this relationship that would last forever: We'd create our very first video on YouTube — or maybe it was

PROMPT:
 A spaceship reached a planet where time flows backwards,


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



SAMPLE 1:
A spaceship reached a planet where time flows backwards, and the ship's captain, who is called "O" by the ship's crew, is sent to retrieve it.

The ship is destroyed, though the ship's crew survive, only to discover that it was the ship's captain, Tux.

Plot

After the crew
Creativity Score: 0.755

SAMPLE 2:
A spaceship reached a planet where time flows backwards, but the ship is stopped by a portal at the end of the galaxy. The portal is shut down, but not before an alien ship is sent to stop it. The alien ship then comes to the planet and destroys it, killing all the human-controlled humans. The aliens then return to the planet
Creativity Score: 0.677

SAMPLE 3:
A spaceship reached a planet where time flows backwards, and, in the process, it destroyed some of the planets in the galaxy.

When the Doctor's spaceship arrives, the planet is completely destroyed by it, but the Doctor's spaceship manages to destroy the planet. The Doctor and the Doctor's spaceship return to the 

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



PIPELINE OUTPUT:
 A spaceship reached a planet where time flows backwards, into the past and into the future. It is a time where the people of Earth have become a little more civilized, and the planet is being controlled by a mysterious force called the "Dark Lords," who have sought to control the Earth as an artificial society.

In this game, the player

MANUAL OUTPUT:
 A spaceship reached a planet where time flows backwards, but it did not come back. It was only after the Enterprise-D took off that this new life began to appear on its own." (Star Trek: Deep Space Nine Companion)
 "We were fortunate enough in having an alien species of space race aboard our ship when we came across one and

CSV Saved: gpt2_clean_experiment.csv

PDF Generated: GPT2_Experiment_Report.pdf
